# Chapter 8: Polyhedra

Source orientation: Hartshorne, *Geometry: Euclid and Beyond*, Chapter 8,
printed pages 435-480 (PDF pages 447-492). Sections 44-47 cover the five
regular solids, Euler's theorem and Cauchy's rigidity theorem, semiregular and
face-regular polyhedra, and symmetry groups of polyhedra.

This notebook is an original, standalone computational lesson for that span.
It uses the PDF only for orientation: no textbook prose, exercise text,
screenshots, page crops, or original figures are reproduced.

## Chapter Goal

Turn the chapter's classification arguments into inspectable data. A
polyhedron will be represented as vertices, polygonal faces, incidence, face
angles, curvature defect, dual faces, and symmetry actions. The goal is not to
replace the synthetic proofs. The goal is to make the bookkeeping visible
enough that the proofs have a working surface.

## Computational Translation Guide

| Chapter idea | Computational object | What we inspect |
| --- | --- | --- |
| Convex polyhedron | finite vertex array plus cyclic face lists | vertices, edges, faces, incidence |
| Regular solid | one face degree and one vertex degree everywhere | edge lengths, face sizes, vertex figures |
| Euler theorem | `V - E + F = 2` | mesh counts and triangulated mesh audits |
| Descartes angle defect | `2*pi - sum(face angles at vertex)` | curvature budget totaling `4*pi` |
| Duality | replace each face by a vertex and each vertex by a face | cube/octahedron and dodecahedron/icosahedron swaps |
| Cauchy rigidity | compare dihedral angles along matching edges | sign-change proof state and Euler contradiction bookkeeping |
| Semiregular solids | vertex configuration such as `(3, 4, 3, 4)` | finite positive-defect candidates and truncation examples |
| Symmetry group | orthogonal maps preserving the vertex set | rotation/full symmetry counts and element-order diagnostics |

## Visualization Storyboard Implemented

1. A Platonic mesh audit shows all five regular solids as countable objects,
   then pairs each with its dual.
2. An interactive Plotly scene lets the reader rotate the five solids and
   compare their mesh counts.
3. An Euler/angle-defect budget turns the regular-solid classification into
   the five possible `(p, q)` pairs.
4. A Cauchy proof-state diagram tracks plus/minus/no-change marks, the
   vertex-figure lemma, and the Euler counting contradiction.
5. A semiregular vertex-configuration chart shows why Archimedean candidates
   are a finite positive-curvature list, aside from the prism and antiprism
   families.
6. A symmetry diagnostic counts rotations and full symmetries directly from
   coordinates, then compares tetrahedral, octahedral, and icosahedral
   rotation groups.

In [ ]:
from pathlib import Path
import itertools
import math
import sys
from dataclasses import dataclass

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import ConvexHull, cKDTree
import sympy as sp
import trimesh

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Euclid and Beyond book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib, save_plotly_html

UNIT = "chapter-08"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
UNIT_ARTIFACT_ROOT = ARTIFACT_ROOT / UNIT
FIGURES = UNIT_ARTIFACT_ROOT / "figures"
INTERACTIVE = UNIT_ARTIFACT_ROOT / "interactive"
TABLES = UNIT_ARTIFACT_ROOT / "tables"
CHECKS = UNIT_ARTIFACT_ROOT / "checks"
for folder in [FIGURES, INTERACTIVE, TABLES, CHECKS]:
    folder.mkdir(parents=True, exist_ok=True)

artifact_records = []


def record_artifact(path, concept, min_bytes=512):
    path = assert_artifact(path, min_bytes=min_bytes)
    if not path.resolve().is_relative_to(UNIT_ARTIFACT_ROOT.resolve()):
        raise AssertionError(f"Artifact escaped chapter-08 subtree: {path}")
    artifact_records.append(
        {
            "concept": concept,
            "path": path.relative_to(BOOK_ROOT).as_posix(),
            "bytes": path.stat().st_size,
        }
    )
    return path


def book_rel(path):
    return Path(path).relative_to(BOOK_ROOT).as_posix()

## Mesh Data For The Five Regular Solids

A drawing of a solid can hide the data that matters. The cells below construct
each solid from explicit coordinates and cyclic face lists. This representation
is deliberately simple: it lets us count edges, build the dual, triangulate for
`trimesh` validation, and ask whether a matrix really preserves the vertex set.

The dodecahedron is built as the dual of the icosahedron. That choice makes
the face/vertex exchange visible instead of treating the two solids as
separate memorized objects.

In [ ]:
@dataclass(frozen=True)
class Polyhedron:
    name: str
    vertices: np.ndarray
    faces: tuple[tuple[int, ...], ...]


def normalize_vertices(vertices):
    vertices = np.asarray(vertices, dtype=float)
    vertices = vertices - vertices.mean(axis=0)
    radius = np.max(np.linalg.norm(vertices, axis=1))
    return vertices / radius


def orient_faces_outward(vertices, faces):
    oriented = []
    for face in faces:
        pts = vertices[list(face)]
        normal = np.cross(pts[1] - pts[0], pts[2] - pts[0])
        if np.dot(normal, pts.mean(axis=0)) < 0:
            face = tuple(reversed(face))
        oriented.append(tuple(face))
    return tuple(oriented)


def make_polyhedron(name, vertices, faces):
    vertices = normalize_vertices(vertices)
    return Polyhedron(name, vertices, orient_faces_outward(vertices, faces))


def tangent_basis(direction):
    direction = np.asarray(direction, dtype=float)
    direction = direction / np.linalg.norm(direction)
    reference = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(reference, direction)) > 0.9:
        reference = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(direction, reference)
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(direction, e1)
    return e1, e2


def poly_edges(poly):
    edges = set()
    for face in poly.faces:
        for a, b in zip(face, face[1:] + face[:1]):
            edges.add(tuple(sorted((a, b))))
    return edges


def triangulated_faces(poly):
    triangles = []
    for face in poly.faces:
        for offset in range(1, len(face) - 1):
            triangles.append((face[0], face[offset], face[offset + 1]))
    return np.array(triangles, dtype=int)


def trimesh_audit(poly):
    mesh = trimesh.Trimesh(vertices=poly.vertices, faces=triangulated_faces(poly), process=False)
    return {
        "trimesh_euler": int(mesh.euler_number),
        "is_watertight": bool(mesh.is_watertight),
        "triangles": int(len(mesh.faces)),
    }


def face_degrees(poly):
    return [len(face) for face in poly.faces]


def vertex_configurations(poly):
    configs = []
    for vertex_index in range(len(poly.vertices)):
        config = sorted(len(face) for face in poly.faces if vertex_index in face)
        configs.append(tuple(config))
    return configs


def edge_length_stats(poly):
    lengths = np.array([np.linalg.norm(poly.vertices[a] - poly.vertices[b]) for a, b in poly_edges(poly)])
    return float(lengths.min()), float(lengths.max()), float(lengths.max() - lengths.min())


def euler_characteristic(poly):
    return len(poly.vertices) - len(poly_edges(poly)) + len(poly.faces)


def dual_polyhedron(poly, name):
    dual_vertices = np.array([poly.vertices[list(face)].mean(axis=0) for face in poly.faces])
    dual_vertices = normalize_vertices(dual_vertices)
    dual_faces = []
    for vertex_index, vertex in enumerate(poly.vertices):
        incident_faces = [face_index for face_index, face in enumerate(poly.faces) if vertex_index in face]
        e1, e2 = tangent_basis(vertex)
        ordered = sorted(
            incident_faces,
            key=lambda face_index: math.atan2(np.dot(dual_vertices[face_index], e2), np.dot(dual_vertices[face_index], e1)),
        )
        dual_faces.append(tuple(ordered))
    return make_polyhedron(name, dual_vertices, dual_faces)


def tetrahedron():
    vertices = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float)
    faces = [(0, 2, 1), (0, 1, 3), (0, 3, 2), (1, 2, 3)]
    return make_polyhedron("tetrahedron", vertices, faces)


def cube():
    vertices = np.array(
        [
            [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
            [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1],
        ],
        dtype=float,
    )
    faces = [(0, 1, 2, 3), (4, 7, 6, 5), (0, 4, 5, 1), (1, 5, 6, 2), (2, 6, 7, 3), (3, 7, 4, 0)]
    return make_polyhedron("cube", vertices, faces)


def octahedron():
    vertices = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]], dtype=float)
    faces = [(0, 2, 4), (2, 1, 4), (1, 3, 4), (3, 0, 4), (2, 0, 5), (1, 2, 5), (3, 1, 5), (0, 3, 5)]
    return make_polyhedron("octahedron", vertices, faces)


def icosahedron():
    phi = (1 + math.sqrt(5)) / 2
    vertices = np.array(
        [
            [-1, phi, 0], [1, phi, 0], [-1, -phi, 0], [1, -phi, 0],
            [0, -1, phi], [0, 1, phi], [0, -1, -phi], [0, 1, -phi],
            [phi, 0, -1], [phi, 0, 1], [-phi, 0, -1], [-phi, 0, 1],
        ],
        dtype=float,
    )
    faces = [
        (0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11),
        (1, 5, 9), (5, 11, 4), (11, 10, 2), (10, 7, 6), (7, 1, 8),
        (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8), (3, 8, 9),
        (4, 9, 5), (2, 4, 11), (6, 2, 10), (8, 6, 7), (9, 8, 1),
    ]
    return make_polyhedron("icosahedron", vertices, faces)


platonic = {
    "tetrahedron": tetrahedron(),
    "cube": cube(),
    "octahedron": octahedron(),
    "icosahedron": icosahedron(),
}
platonic["dodecahedron"] = dual_polyhedron(platonic["icosahedron"], "dodecahedron")
platonic_order = ["tetrahedron", "cube", "octahedron", "dodecahedron", "icosahedron"]
platonic = {name: platonic[name] for name in platonic_order}

dual_names = {
    "tetrahedron": "tetrahedron",
    "cube": "octahedron",
    "octahedron": "cube",
    "dodecahedron": "icosahedron",
    "icosahedron": "dodecahedron",
}

rows = []
for name, poly in platonic.items():
    _, _, edge_spread = edge_length_stats(poly)
    audit = trimesh_audit(poly)
    config_counts = pd.Series(vertex_configurations(poly)).value_counts().to_dict()
    rows.append(
        {
            "solid": name,
            "V": len(poly.vertices),
            "E": len(poly_edges(poly)),
            "F": len(poly.faces),
            "V_minus_E_plus_F": euler_characteristic(poly),
            "face_degrees": sorted(set(face_degrees(poly))),
            "vertex_config": sorted(config_counts, key=str)[0],
            "dual": dual_names[name],
            "edge_length_spread": edge_spread,
            "trimesh_euler": audit["trimesh_euler"],
            "trimesh_watertight": audit["is_watertight"],
        }
    )

platonic_df = pd.DataFrame(rows)
platonic_table_path = TABLES / "platonic_euler_duality_invariants.csv"
platonic_df.to_csv(platonic_table_path, index=False)
record_artifact(platonic_table_path, "Platonic Euler and duality invariant table", min_bytes=128)
platonic_df

The table is the first classification checkpoint. Every row has Euler
characteristic `2`, one face degree, one vertex configuration, and a dual whose
vertex and face counts are exchanged. The edge-length spread column should be
numerically zero for these coordinate models.

In [ ]:
def add_poly_collection(ax, poly, facecolor, edgecolor="#1f2937", alpha=0.74):
    polygons = [[poly.vertices[i] for i in face] for face in poly.faces]
    collection = Poly3DCollection(polygons, facecolor=facecolor, edgecolor=edgecolor, linewidth=0.8, alpha=alpha)
    ax.add_collection3d(collection)
    for a, b in poly_edges(poly):
        segment = poly.vertices[[a, b]]
        ax.plot(segment[:, 0], segment[:, 1], segment[:, 2], color=edgecolor, linewidth=0.8)
    ax.scatter(poly.vertices[:, 0], poly.vertices[:, 1], poly.vertices[:, 2], s=10, color="#991b1b", depthshade=False)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_zlim(-1.05, 1.05)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=22, azim=35)


palette = {
    "tetrahedron": "#8ecae6",
    "cube": "#f4a261",
    "octahedron": "#90be6d",
    "dodecahedron": "#cdb4db",
    "icosahedron": "#f6bd60",
}

fig = plt.figure(figsize=(14, 8.5))
for index, name in enumerate(platonic_order, start=1):
    poly = platonic[name]
    ax = fig.add_subplot(2, 3, index, projection="3d")
    add_poly_collection(ax, poly, palette[name])
    ax.set_title(
        f"{name}\nV={len(poly.vertices)}, E={len(poly_edges(poly))}, F={len(poly.faces)}, dual={dual_names[name]}",
        fontsize=10,
        pad=2,
    )
fig.text(0.02, 0.96, "Platonic solids as incidence data", fontsize=16, weight="bold")
fig.text(0.02, 0.925, "Inspection target: Euler counts, one local vertex configuration, and the face/vertex swap under duality.", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.91))
platonic_figure_path = save_matplotlib(fig, UNIT, "figures", "platonic_euler_duality_diagnostics.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(platonic_figure_path, "Platonic mesh diagnostics and duality", min_bytes=2048)


def plotly_mesh_trace(poly, color, name):
    triangles = triangulated_faces(poly)
    return go.Mesh3d(
        x=poly.vertices[:, 0],
        y=poly.vertices[:, 1],
        z=poly.vertices[:, 2],
        i=triangles[:, 0],
        j=triangles[:, 1],
        k=triangles[:, 2],
        color=color,
        opacity=0.72,
        name=name,
        hovertemplate=f"{name}<br>V={len(poly.vertices)} E={len(poly_edges(poly))} F={len(poly.faces)}<extra></extra>",
    )


fig3d = make_subplots(
    rows=2,
    cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}], [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[name.title() for name in platonic_order] + ["Dual map"],
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)
positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]
for (row, col), name in zip(positions, platonic_order):
    fig3d.add_trace(plotly_mesh_trace(platonic[name], palette[name], name), row=row, col=col)

for offset, (name, color) in enumerate([("cube", "#f4a261"), ("octahedron", "#90be6d"), ("dodecahedron", "#cdb4db"), ("icosahedron", "#f6bd60")]):
    shifted = Polyhedron(name, platonic[name].vertices + np.array([1.7 * (offset - 1.5), 0, 0]), platonic[name].faces)
    fig3d.add_trace(plotly_mesh_trace(shifted, color, name), row=2, col=3)

for scene_index in range(1, 7):
    fig3d.update_scenes(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, row=(scene_index - 1) // 3 + 1, col=(scene_index - 1) % 3 + 1)
fig3d.update_layout(
    title="Interactive Platonic solids: rotate to inspect faces, edges, vertices, and dual pairs",
    template="plotly_white",
    height=850,
    showlegend=False,
    margin={"l": 0, "r": 0, "t": 90, "b": 0},
)
platonic_html_path = save_plotly_html(fig3d, UNIT, "interactive", "platonic_solids_duality.html", root=ARTIFACT_ROOT, include_plotlyjs="cdn")
record_artifact(platonic_html_path, "Interactive Platonic solids and duality", min_bytes=4096)

display_artifact(platonic_figure_path, width=980)
display_artifact(platonic_html_path, width="100%", height=720)

## Euler Characteristic And Angle Defect

The regular-solid classification becomes finite before any model is drawn. If
`q` regular `p`-gons meet at a convex vertex, then the planar face angles
around that vertex must sum to less than `2*pi`:

$$q\frac{(p-2)\pi}{p} < 2\pi.$$

The exact integer solutions with `p >= 3` and `q >= 3` are the five Platonic
vertex types. The same computation also explains Descartes' defect theorem:
each convex vertex has positive defect, and the total defect of the whole
polyhedron is `4*pi`.

In [ ]:
p, q = sp.symbols("p q", integer=True, positive=True)
regular_pair_candidates = []
for p_value in range(3, 21):
    for q_value in range(3, 21):
        if q_value * (p_value - 2) < 2 * p_value:
            regular_pair_candidates.append((p_value, q_value))
assert regular_pair_candidates == [(3, 3), (3, 4), (3, 5), (4, 3), (5, 3)]

pair_to_solid = {(3, 3): "tetrahedron", (4, 3): "cube", (3, 4): "octahedron", (5, 3): "dodecahedron", (3, 5): "icosahedron"}
angle_rows = []
for p_value, q_value in regular_pair_candidates:
    face_angle = (p_value - 2) * math.pi / p_value
    vertex_defect = 2 * math.pi - q_value * face_angle
    solid = pair_to_solid[(p_value, q_value)]
    V = len(platonic[solid].vertices)
    angle_rows.append(
        {
            "p_gon": p_value,
            "q_around_vertex": q_value,
            "solid": solid,
            "angle_sum_over_pi": q_value * (p_value - 2) / p_value,
            "defect_over_pi": vertex_defect / math.pi,
            "total_defect_over_pi": V * vertex_defect / math.pi,
        }
    )
angle_df = pd.DataFrame(angle_rows)

all_grid = []
for p_value in range(3, 9):
    for q_value in range(3, 7):
        all_grid.append(
            {
                "p": p_value,
                "q": q_value,
                "defect_over_pi": 2 - q_value * (p_value - 2) / p_value,
                "possible": (p_value, q_value) in regular_pair_candidates,
            }
        )
grid_df = pd.DataFrame(all_grid)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.5, 5.2), gridspec_kw={"width_ratios": [1.2, 1]})
piv = grid_df.pivot(index="q", columns="p", values="defect_over_pi")
im = ax0.imshow(piv.values, cmap="RdYlGn", vmin=-1.0, vmax=1.0, origin="lower")
ax0.set_xticks(range(len(piv.columns)), labels=piv.columns)
ax0.set_yticks(range(len(piv.index)), labels=piv.index)
ax0.set_xlabel("face size p")
ax0.set_ylabel("faces at a vertex q")
ax0.set_title("Positive defect marks possible regular vertices")
for y, q_value in enumerate(piv.index):
    for x, p_value in enumerate(piv.columns):
        value = piv.loc[q_value, p_value]
        label = f"{value:.2g}"
        if (p_value, q_value) in pair_to_solid:
            label += "\n" + pair_to_solid[(p_value, q_value)].replace("hedron", ".")
        ax0.text(x, y, label, ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04, label="defect / pi")

bars = ax1.bar(angle_df["solid"], angle_df["defect_over_pi"], color=[palette[name] for name in angle_df["solid"]], edgecolor="#1f2937")
ax1.axhline(0, color="#1f2937", linewidth=0.8)
ax1.set_ylabel("single-vertex defect / pi")
ax1.set_title("Each solid spends a total defect budget of 4*pi")
ax1.tick_params(axis="x", rotation=25)
for bar, total in zip(bars, angle_df["total_defect_over_pi"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03, f"total={total:.0f}pi", ha="center", fontsize=8)
fig.suptitle("Euler and Descartes bookkeeping for regular solids", x=0.02, ha="left", fontsize=15, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
angle_path = save_matplotlib(fig, UNIT, "figures", "euler_angle_defect_budget.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(angle_path, "Euler angle-defect budget", min_bytes=2048)

angle_table_path = TABLES / "platonic_angle_defects.csv"
angle_df.to_csv(angle_table_path, index=False)
record_artifact(angle_table_path, "Platonic angle defect table", min_bytes=128)

display_artifact(angle_path, width=980)
angle_df

The heatmap is the finite decision space for regular convex vertices. Green
means positive defect. Only five integer cells survive, so the five Platonic
solids are already forced locally. The bar plot then checks the global budget:
number of vertices times defect equals `4*pi` in every case.

## Cauchy Rigidity As A Proof-State Diagram

Cauchy's theorem says that two convex polyhedra with congruent corresponding
faces, arranged with the same incidence, are congruent. The proof studies what
happens if corresponding dihedral angles do not all agree.

The diagnostic below does not prove the theorem by computation. Instead, it
exposes the proof state. Edges are marked `+`, `-`, or `0` depending on whether
a dihedral angle is larger, smaller, or equal in the comparison. Around each
vertex, the vertex figure translates those edge marks into a cyclic sign
pattern. If any changes remain, the sign-change lemma forces enough changes
that Euler's formula creates a counting contradiction. The only stable state
is all zeros.

In [ ]:
proof_graph = nx.DiGraph()
proof_edges = [
    ("congruent faces", "compare dihedral angles"),
    ("same incidence", "compare dihedral angles"),
    ("compare dihedral angles", "mark edges +, -, 0"),
    ("mark edges +, -, 0", "vertex figure cycles"),
    ("vertex figure cycles", "at least four sign changes"),
    ("at least four sign changes", "count changes two ways"),
    ("Euler formula", "count changes two ways"),
    ("count changes two ways", "contradiction unless all marks vanish"),
    ("contradiction unless all marks vanish", "dihedral angles agree"),
    ("dihedral angles agree", "polyhedra congruent"),
]
proof_graph.add_edges_from(proof_edges)

cycle_signs = ["+", "+", "0", "-", "-", "+", "0", "-"]
visible_signs = [s for s in cycle_signs if s != "0"]
sign_changes = sum(a != b for a, b in zip(visible_signs, visible_signs[1:] + visible_signs[:1]))
assert sign_changes == 4

fig = plt.figure(figsize=(15.5, 6.6))
grid = fig.add_gridspec(1, 2, width_ratios=[1, 1.55])
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])

n = len(cycle_signs)
angles = np.linspace(0, 2 * math.pi, n, endpoint=False) + math.pi / 10
coords = np.column_stack([np.cos(angles), np.sin(angles)])
for i in range(n):
    a = coords[i]
    b = coords[(i + 1) % n]
    ax0.plot([a[0], b[0]], [a[1], b[1]], color="#334155", linewidth=2)
for i, (xy, sign) in enumerate(zip(coords, cycle_signs)):
    color = {"+": "#dc2626", "-": "#2563eb", "0": "#64748b"}[sign]
    ax0.scatter([xy[0]], [xy[1]], s=600, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax0.text(xy[0], xy[1], sign, ha="center", va="center", color="white", fontsize=16, weight="bold")
    ax0.text(1.08 * xy[0], 1.08 * xy[1], f"e{i+1}", ha="center", va="center", fontsize=8)
ax0.text(0, -1.42, f"Ignoring zeros, this sample has {sign_changes} cyclic sign changes.", ha="center", fontsize=10)
ax0.set_title("Vertex figure mark cycle", pad=18)
ax0.set_aspect("equal")
ax0.axis("off")

pos = {
    "congruent faces": (0.0, 4.0),
    "same incidence": (0.0, 2.5),
    "compare dihedral angles": (2.3, 3.25),
    "mark edges +, -, 0": (4.7, 3.25),
    "vertex figure cycles": (7.1, 3.25),
    "at least four sign changes": (9.6, 3.25),
    "Euler formula": (7.1, 1.25),
    "count changes two ways": (9.6, 1.95),
    "contradiction unless all marks vanish": (12.3, 1.95),
    "dihedral angles agree": (14.9, 1.95),
    "polyhedra congruent": (17.3, 1.95),
}
labels = {
    "congruent faces": "congruent\nfaces",
    "same incidence": "same\nincidence",
    "compare dihedral angles": "compare\ndihedral\nangles",
    "mark edges +, -, 0": "mark edges\n+, -, 0",
    "vertex figure cycles": "vertex\nfigure\ncycles",
    "at least four sign changes": "at least four\nsign changes",
    "Euler formula": "Euler\nformula",
    "count changes two ways": "count changes\ntwo ways",
    "contradiction unless all marks vanish": "contradiction\nunless all\nmarks vanish",
    "dihedral angles agree": "dihedral\nangles\nagree",
    "polyhedra congruent": "polyhedra\ncongruent",
}
node_colors = ["#fde68a" if "Euler" in node else "#bfdbfe" if "sign" in node or "mark" in node or "contradiction" in node else "#dcfce7" for node in proof_graph.nodes]
nx.draw_networkx_nodes(proof_graph, pos, ax=ax1, node_color=node_colors, edgecolors="#1f2937", node_size=2600)
nx.draw_networkx_labels(proof_graph, pos, labels=labels, ax=ax1, font_size=7.8)
nx.draw_networkx_edges(proof_graph, pos, ax=ax1, arrows=True, arrowstyle="-|>", arrowsize=13, width=1.15, edge_color="#475569", connectionstyle="arc3,rad=0.03")
ax1.set_xlim(-0.8, 18.2)
ax1.set_ylim(0.25, 4.65)
ax1.set_title("Dependency graph for Cauchy's proof state")
ax1.axis("off")

fig.suptitle("Cauchy rigidity: what the proof has to rule out", x=0.02, ha="left", fontsize=15, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
cauchy_path = save_matplotlib(fig, UNIT, "figures", "cauchy_rigidity_sign_state.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(cauchy_path, "Cauchy rigidity sign-change proof state", min_bytes=2048)

cauchy_count_checks = []
for name, poly in platonic.items():
    V = len(poly.vertices)
    face_degree_counts = pd.Series(face_degrees(poly)).value_counts().to_dict()
    upper = 2 * face_degree_counts.get(3, 0) + sum(n * count for n, count in face_degree_counts.items() if n >= 4)
    lower = 4 * V
    cauchy_count_checks.append({"solid": name, "4V": int(lower), "face_count_upper_bound": int(upper), "all_edges_marked_contradiction": bool(lower > upper)})
cauchy_count_df = pd.DataFrame(cauchy_count_checks)
cauchy_table_path = TABLES / "cauchy_counting_diagnostics.csv"
cauchy_count_df.to_csv(cauchy_table_path, index=False)
record_artifact(cauchy_table_path, "Cauchy counting diagnostics", min_bytes=128)

display_artifact(cauchy_path, width=980)
cauchy_count_df


The table gives a deliberately narrow computational echo of the proof. If
every edge were marked as changing, the vertex-figure lemma would demand at
least `4V` sign changes. Counting by faces gives the displayed upper bound.
For these regular examples, the lower bound already exceeds the upper bound.
The full theorem needs the more careful net argument for mixed marked and
unmarked edges, but the obstruction is the same kind of Euler bookkeeping.

## Semiregular And Truncated Polyhedra

The semiregular part of the chapter relaxes regularity: faces are still
regular polygons, but more than one face size may occur around a vertex. A
vertex configuration such as `(3, 4, 3, 4)` records the cyclic face sizes
encountered around each vertex.

The local convexity test is again an angle budget. For a configuration
`(a_1, ..., a_k)`, positive defect is equivalent to

$$\sum_i \frac{2}{a_i} > k - 2.$$

This table includes the Platonic cases, selected Archimedean cases, and the
two infinite families of prisms and antiprisms. The chart uses defect rather
than a rendered catalog because the classification proof is driven by this
finite budget.

In [ ]:
semiregular_rows = [
    ("tetrahedron", (3, 3, 3), "Platonic", 4),
    ("cube", (4, 4, 4), "Platonic", 8),
    ("octahedron", (3, 3, 3, 3), "Platonic", 6),
    ("dodecahedron", (5, 5, 5), "Platonic", 20),
    ("icosahedron", (3, 3, 3, 3, 3), "Platonic", 12),
    ("n-gonal prism", (4, 4, 6), "Prism family sample n=6", 12),
    ("n-gonal antiprism", (3, 3, 3, 6), "Antiprism family sample n=6", 12),
    ("truncated tetrahedron", (3, 6, 6), "Truncated", 12),
    ("truncated cube", (3, 8, 8), "Truncated", 24),
    ("truncated octahedron", (4, 6, 6), "Truncated", 24),
    ("truncated dodecahedron", (3, 10, 10), "Truncated", 60),
    ("truncated icosahedron", (5, 6, 6), "Truncated", 60),
    ("cuboctahedron", (3, 4, 3, 4), "Rectified", 12),
    ("icosidodecahedron", (3, 5, 3, 5), "Rectified", 30),
    ("rhombicuboctahedron", (3, 4, 4, 4), "Expanded", 24),
    ("rhombicosidodecahedron", (3, 4, 5, 4), "Expanded", 60),
    ("truncated cuboctahedron", (4, 6, 8), "Truncated", 48),
    ("truncated icosidodecahedron", (4, 6, 10), "Truncated", 120),
    ("snub cube", (3, 3, 3, 3, 4), "Snub", 24),
    ("snub dodecahedron", (3, 3, 3, 3, 5), "Snub", 60),
]


def defect_for_config(config):
    return 2 * math.pi - sum((n - 2) * math.pi / n for n in config)


semiregular_df = pd.DataFrame(
    [
        {
            "name": name,
            "vertex_config": config,
            "kind": kind,
            "V": V,
            "faces_at_vertex": len(config),
            "defect_over_pi": defect_for_config(config) / math.pi,
            "total_defect_over_pi": V * defect_for_config(config) / math.pi,
            "positive_defect": defect_for_config(config) > 0,
        }
        for name, config, kind, V in semiregular_rows
    ]
)
assert semiregular_df["positive_defect"].all()
assert np.allclose(semiregular_df["total_defect_over_pi"], 4.0)

semiregular_table_path = TABLES / "semiregular_vertex_configurations.csv"
semiregular_df.to_csv(semiregular_table_path, index=False)
record_artifact(semiregular_table_path, "Semiregular vertex configuration table", min_bytes=256)

kind_colors = {
    "Platonic": "#2563eb",
    "Prism family sample n=6": "#16a34a",
    "Antiprism family sample n=6": "#15803d",
    "Truncated": "#ea580c",
    "Rectified": "#7c3aed",
    "Expanded": "#be123c",
    "Snub": "#0891b2",
}
plot_df = semiregular_df.copy()
plot_df["label"] = plot_df.apply(lambda row: f"{row['name']}  {row['vertex_config']}", axis=1)
plot_df = plot_df.sort_values(["faces_at_vertex", "defect_over_pi", "name"], ascending=[True, False, True]).reset_index(drop=True)
plot_df["row"] = np.arange(len(plot_df))[::-1]

fig, ax = plt.subplots(figsize=(13.5, 9.2))
for kind, group in plot_df.groupby("kind", sort=False):
    ax.scatter(
        group["defect_over_pi"],
        group["row"],
        s=80 + group["V"] * 2.4,
        color=kind_colors[kind],
        alpha=0.86,
        label=kind,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
for _, row in plot_df.iterrows():
    ax.text(row["defect_over_pi"] + 0.018, row["row"], f"k={row['faces_at_vertex']}", va="center", fontsize=8, color="#334155")
ax.axvline(0, color="#1f2937", linewidth=1)
ax.set_yticks(plot_df["row"], labels=plot_df["label"])
ax.set_xlabel("single-vertex defect / pi")
ax.set_title("Semiregular candidates remain finite because every vertex spends positive defect")
ax.set_xlim(-0.02, 1.12)
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right", fontsize=8, title="configuration family")
fig.text(0.02, 0.965, "Inspection target: every listed configuration lies to the right of zero and its total defect is 4*pi.", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.95))
semiregular_path = save_matplotlib(fig, UNIT, "figures", "semiregular_vertex_configurations.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(semiregular_path, "Semiregular positive-defect vertex configurations", min_bytes=2048)

display_artifact(semiregular_path, width=980)
semiregular_df[["name", "vertex_config", "kind", "V", "defect_over_pi", "total_defect_over_pi"]]


### A Truncation Lab

Truncation is a good computational bridge between regular and semiregular
solids. Cutting a vertex creates a new face; the old faces survive with more
sides. The exact cut distance matters if the result is to have all new edges
equal, but the incidence pattern is already visible in the mesh.

The truncated tetrahedron below is the cleanest example: cutting each edge one
third of the way from each endpoint turns four triangular faces into four
hexagons and creates four new triangles. The vertex configuration becomes
`(3, 6, 6)`.

In [ ]:
def truncate_polyhedron(poly, t=1 / 3, name=None):
    new_vertices = []
    vertex_for_oriented_edge = {}

    def point_on_oriented_edge(a, b):
        key = (a, b)
        if key not in vertex_for_oriented_edge:
            vertex_for_oriented_edge[key] = len(new_vertices)
            new_vertices.append((1 - t) * poly.vertices[a] + t * poly.vertices[b])
        return vertex_for_oriented_edge[key]

    new_faces = []
    for face in poly.faces:
        truncated_face = []
        n = len(face)
        for i, a in enumerate(face):
            b = face[(i + 1) % n]
            truncated_face.extend([point_on_oriented_edge(a, b), point_on_oriented_edge(b, a)])
        new_faces.append(tuple(truncated_face))

    neighbors = {i: set() for i in range(len(poly.vertices))}
    for a, b in poly_edges(poly):
        neighbors[a].add(b)
        neighbors[b].add(a)

    new_vertices_array = np.array(new_vertices)
    for a, neighbor_set in neighbors.items():
        e1, e2 = tangent_basis(poly.vertices[a])
        ids = [point_on_oriented_edge(a, b) for b in neighbor_set]
        ordered = sorted(ids, key=lambda idx: math.atan2(np.dot(new_vertices_array[idx], e2), np.dot(new_vertices_array[idx], e1)))
        new_faces.append(tuple(ordered))

    return make_polyhedron(name or f"truncated {poly.name}", np.array(new_vertices), new_faces)


def hull_grouped_polyhedron(name, vertices, tolerance=1e-7):
    vertices = normalize_vertices(vertices)
    hull = ConvexHull(vertices)
    used = np.zeros(len(hull.simplices), dtype=bool)
    faces = []
    for i, equation in enumerate(hull.equations):
        if used[i]:
            continue
        normal = equation[:3]
        offset = equation[3]
        group = [i]
        used[i] = True
        for j, other in enumerate(hull.equations):
            if not used[j] and np.allclose(other[:3], normal, atol=tolerance) and abs(other[3] - offset) < tolerance:
                used[j] = True
                group.append(j)
        face_vertices = sorted({int(v) for simplex_index in group for v in hull.simplices[simplex_index]})
        center = vertices[face_vertices].mean(axis=0)
        e1, e2 = tangent_basis(normal)
        ordered = sorted(face_vertices, key=lambda idx: math.atan2(np.dot(vertices[idx] - center, e2), np.dot(vertices[idx] - center, e1)))
        faces.append(tuple(ordered))
    return make_polyhedron(name, vertices, faces)


def cuboctahedron():
    vertices = []
    for zero_coordinate in range(3):
        active = [axis for axis in range(3) if axis != zero_coordinate]
        for first_sign in [-1, 1]:
            for second_sign in [-1, 1]:
                vertex = [0.0, 0.0, 0.0]
                vertex[active[0]] = first_sign
                vertex[active[1]] = second_sign
                vertices.append(vertex)
    return hull_grouped_polyhedron("cuboctahedron", np.array(vertices))


truncated_tetra = truncate_polyhedron(platonic["tetrahedron"], t=1 / 3, name="truncated tetrahedron")
cubocta = cuboctahedron()
semiregular_examples = [truncated_tetra, cubocta]

lab_rows = []
for poly in semiregular_examples:
    _, _, spread = edge_length_stats(poly)
    lab_rows.append(
        {
            "solid": poly.name,
            "V": len(poly.vertices),
            "E": len(poly_edges(poly)),
            "F": len(poly.faces),
            "Euler": euler_characteristic(poly),
            "face_degree_counts": dict(pd.Series(face_degrees(poly)).value_counts().sort_index()),
            "vertex_configurations": dict(pd.Series(vertex_configurations(poly)).value_counts()),
            "edge_length_spread": spread,
            "trimesh_watertight": trimesh_audit(poly)["is_watertight"],
        }
    )
truncation_lab_df = pd.DataFrame(lab_rows)
truncation_table_path = TABLES / "truncation_lab_invariants.csv"
truncation_lab_df.to_csv(truncation_table_path, index=False)
record_artifact(truncation_table_path, "Truncation lab invariants", min_bytes=128)

fig = plt.figure(figsize=(11.5, 5.5))
for index, poly in enumerate(semiregular_examples, start=1):
    ax = fig.add_subplot(1, 2, index, projection="3d")
    add_poly_collection(ax, poly, "#a7f3d0" if index == 1 else "#ddd6fe")
    config = sorted(pd.Series(vertex_configurations(poly)).value_counts().to_dict(), key=str)[0]
    ax.set_title(f"{poly.name}\nconfig {config}; V-E+F={euler_characteristic(poly)}", fontsize=10)
fig.suptitle("Truncation and rectification examples", x=0.02, ha="left", fontsize=15, weight="bold")
fig.text(0.02, 0.91, "Inspection target: new cut faces, old faces with more sides, and unchanged Euler characteristic.", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.88))
truncation_path = save_matplotlib(fig, UNIT, "figures", "truncated_semiregular_examples.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(truncation_path, "Truncated and semiregular mesh examples", min_bytes=2048)

display_artifact(truncation_path, width=900)
truncation_lab_df

The cuboctahedron is included as a rectification example: it can be seen by
cutting a cube or an octahedron far enough to leave triangles and squares.
Together, the lab examples separate two questions that are often blurred: the
Euler/incidence type is combinatorial, while true semiregularity also asks for
metric equality of the new edges and angles.

## Symmetry Group Diagnostics

The chapter's last section identifies rotations of regular solids with
familiar finite groups. We can test the group size directly from coordinates:
enumerate orthogonal matrices that send the vertex set to itself, then separate
orientation-preserving rotations from orientation-reversing symmetries.

This diagnostic is intentionally concrete. It does not assume the answer is
`A_4`, `S_4`, or `A_5`; it counts the actual vertex-set symmetries and then
compares the result with those group names.

In [ ]:
def symmetry_matrices(poly, tolerance=1e-6):
    vertices = normalize_vertices(poly.vertices)
    vertex_count = len(vertices)
    base = None
    for combination in itertools.combinations(range(vertex_count), 3):
        candidate = np.column_stack([vertices[i] for i in combination])
        if abs(np.linalg.det(candidate)) > 1e-3:
            base = combination
            break
    if base is None:
        raise ValueError(f"Could not find a noncoplanar vertex triple for {poly.name}")

    source = np.column_stack([vertices[i] for i in base])
    source_gram = source.T @ source
    inverse_source = np.linalg.inv(source)
    tree = cKDTree(vertices)
    matrices = {}
    for target in itertools.permutations(range(vertex_count), 3):
        target_matrix = np.column_stack([vertices[i] for i in target])
        if not np.allclose(target_matrix.T @ target_matrix, source_gram, atol=tolerance):
            continue
        matrix = target_matrix @ inverse_source
        mapped = vertices @ matrix.T
        distances, _ = tree.query(mapped, k=1)
        if float(distances.max()) < 1e-5 and np.allclose(matrix.T @ matrix, np.eye(3), atol=1e-5):
            matrices[tuple(np.round(matrix, 6).ravel())] = matrix
    return list(matrices.values())


def matrix_order(matrix, max_power=12):
    current = np.eye(3)
    for order in range(1, max_power + 1):
        current = current @ matrix
        if np.allclose(current, np.eye(3), atol=1e-5):
            return order
    return None


symmetry_rows = []
element_order_rows = []
expected_rotation_orders = {"tetrahedron": 12, "cube": 24, "octahedron": 24, "dodecahedron": 60, "icosahedron": 60}
for name in platonic_order:
    poly = platonic[name]
    matrices = symmetry_matrices(poly)
    rotations = [matrix for matrix in matrices if np.linalg.det(matrix) > 0]
    full_order = len(matrices)
    rotation_order = len(rotations)
    symmetry_rows.append(
        {
            "solid": name,
            "rotation_group": {"tetrahedron": "T ~= A4", "cube": "O ~= S4", "octahedron": "O ~= S4", "dodecahedron": "I ~= A5", "icosahedron": "I ~= A5"}[name],
            "rotation_order": rotation_order,
            "full_symmetry_order": full_order,
            "expected_rotation_order": expected_rotation_orders[name],
        }
    )
    counts = pd.Series([matrix_order(matrix, max_power=10) for matrix in rotations]).value_counts().sort_index()
    for order, count in counts.items():
        element_order_rows.append({"solid": name, "element_order": int(order), "rotation_count": int(count)})

symmetry_df = pd.DataFrame(symmetry_rows)
element_orders_df = pd.DataFrame(element_order_rows)
assert all(symmetry_df["rotation_order"] == symmetry_df["expected_rotation_order"])
assert dict(zip(symmetry_df["solid"], symmetry_df["full_symmetry_order"])) == {
    "tetrahedron": 24,
    "cube": 48,
    "octahedron": 48,
    "dodecahedron": 120,
    "icosahedron": 120,
}

symmetry_table_path = TABLES / "symmetry_group_counts.csv"
element_order_table_path = TABLES / "rotation_element_orders.csv"
symmetry_df.to_csv(symmetry_table_path, index=False)
element_orders_df.to_csv(element_order_table_path, index=False)
record_artifact(symmetry_table_path, "Symmetry group counts", min_bytes=128)
record_artifact(element_order_table_path, "Rotation element order counts", min_bytes=128)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.5, 5.4), gridspec_kw={"width_ratios": [1, 1.2]})
width = 0.36
x = np.arange(len(symmetry_df))
ax0.bar(x - width / 2, symmetry_df["rotation_order"], width=width, color="#38bdf8", edgecolor="#0f172a", label="rotations")
ax0.bar(x + width / 2, symmetry_df["full_symmetry_order"], width=width, color="#f97316", edgecolor="#0f172a", label="full symmetries")
ax0.set_xticks(x, labels=symmetry_df["solid"], rotation=25, ha="right")
ax0.set_ylabel("group order")
ax0.set_title("Coordinate count of vertex-set symmetries")
ax0.legend()

for solid, group in element_orders_df.groupby("solid"):
    ax1.plot(group["element_order"], group["rotation_count"], marker="o", linewidth=2, label=solid)
ax1.set_xlabel("rotation element order")
ax1.set_ylabel("number of rotations")
ax1.set_title("Element-order fingerprint of each rotation group")
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.grid(True, alpha=0.25)
ax1.legend(fontsize=8)
fig.suptitle("Symmetry diagnostics for Platonic solids", x=0.02, ha="left", fontsize=15, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
symmetry_path = save_matplotlib(fig, UNIT, "figures", "symmetry_group_diagnostics.png", root=ARTIFACT_ROOT, dpi=180)
plt.close(fig)
record_artifact(symmetry_path, "Platonic symmetry group diagnostics", min_bytes=2048)

display_artifact(symmetry_path, width=980)
symmetry_df

The element-order fingerprint distinguishes the rotation groups. The
tetrahedron has rotations of orders `1`, `2`, and `3` and order `12` overall.
The cube and octahedron share the same rotation group of order `24`. The
dodecahedron and icosahedron share the icosahedral rotation group of order
`60`, with the `1, 24, 20, 15` split by orders `1, 5, 3, 2` that the source
section uses when identifying the group with `A_5`.

## Applied Lab: Perturb The Truncation Cut

The exact semiregular cut of a polyhedron is metric, not merely topological.
For a tetrahedron, the one-third cut is special because the new triangles and
the surviving hexagons have the same edge length. The lab below changes the
cut parameter and records what stays fixed and what changes.

In [ ]:
perturbation_rows = []
for t in [0.18, 0.25, 1 / 3, 0.42]:
    poly = truncate_polyhedron(platonic["tetrahedron"], t=t, name=f"truncated tetrahedron t={t:.3f}")
    lengths = np.array([np.linalg.norm(poly.vertices[a] - poly.vertices[b]) for a, b in poly_edges(poly)])
    perturbation_rows.append(
        {
            "cut_fraction": t,
            "V": len(poly.vertices),
            "E": len(poly_edges(poly)),
            "F": len(poly.faces),
            "Euler": euler_characteristic(poly),
            "min_edge": float(lengths.min()),
            "max_edge": float(lengths.max()),
            "edge_spread": float(lengths.max() - lengths.min()),
            "all_edges_equal": bool(np.allclose(lengths, lengths[0], atol=1e-8)),
        }
    )
perturbation_df = pd.DataFrame(perturbation_rows)
assert perturbation_df["Euler"].eq(2).all()
assert perturbation_df.loc[np.isclose(perturbation_df["cut_fraction"], 1 / 3), "all_edges_equal"].iloc[0]

perturbation_table_path = TABLES / "truncation_parameter_lab.csv"
perturbation_df.to_csv(perturbation_table_path, index=False)
record_artifact(perturbation_table_path, "Truncation parameter lab", min_bytes=128)
perturbation_df

The invariant column and the metric column tell different stories.
`V - E + F` remains `2` for every cut in the lab, but only the one-third cut
makes all edges equal. That is why a visual catalog of semiregular solids needs
both incidence bookkeeping and metric checks.

## Final Sanity Checks

The final cell collects the chapter's core identities and artifact checks. It
verifies Euler characteristic, dual count swaps, Descartes total defect,
semiregular positive defect, symmetry group orders, artifact paths, and nonzero
artifact sizes.

In [ ]:
identity_checks = {}
identity_checks["platonic_euler"] = {name: int(euler_characteristic(poly)) for name, poly in platonic.items()}
identity_checks["dual_count_swaps"] = {}
for name, dual_name in dual_names.items():
    poly = platonic[name]
    dual = platonic[dual_name]
    identity_checks["dual_count_swaps"][name] = {
        "V_equals_dual_F": len(poly.vertices) == len(dual.faces),
        "F_equals_dual_V": len(poly.faces) == len(dual.vertices),
        "E_equals_dual_E": len(poly_edges(poly)) == len(poly_edges(dual)),
    }
identity_checks["edge_length_spreads"] = {name: float(edge_length_stats(poly)[2]) for name, poly in platonic.items()}
identity_checks["descartes_total_defect_over_pi"] = dict(zip(angle_df["solid"], angle_df["total_defect_over_pi"].round(10)))
identity_checks["regular_pair_candidates"] = regular_pair_candidates
identity_checks["semiregular_total_defect_over_pi"] = {
    row["name"]: float(round(row["total_defect_over_pi"], 10)) for _, row in semiregular_df.iterrows()
}
identity_checks["symmetry_orders"] = {
    row["solid"]: {"rotation_order": int(row["rotation_order"]), "full_symmetry_order": int(row["full_symmetry_order"])}
    for _, row in symmetry_df.iterrows()
}
identity_checks["truncation_lab_euler"] = dict(zip(perturbation_df["cut_fraction"].round(6).astype(str), perturbation_df["Euler"].astype(int)))

assert set(identity_checks["platonic_euler"].values()) == {2}
assert all(all(values.values()) for values in identity_checks["dual_count_swaps"].values())
assert max(identity_checks["edge_length_spreads"].values()) < 1e-8
assert np.allclose(list(identity_checks["descartes_total_defect_over_pi"].values()), 4.0)
assert regular_pair_candidates == [(3, 3), (3, 4), (3, 5), (4, 3), (5, 3)]
assert np.allclose(list(identity_checks["semiregular_total_defect_over_pi"].values()), 4.0)
assert dict(zip(symmetry_df["solid"], symmetry_df["rotation_order"])) == expected_rotation_orders
assert perturbation_df["Euler"].eq(2).all()

artifact_checks = []
seen_paths = set()
for record in artifact_records:
    path = BOOK_ROOT / record["path"]
    assert path.resolve().is_relative_to(UNIT_ARTIFACT_ROOT.resolve())
    assert path.relative_to(BOOK_ROOT).as_posix().startswith("artifacts/chapter-08/")
    assert_artifact(path, min_bytes=128)
    seen_paths.add(record["path"])
    artifact_checks.append({**record, "exists": True, "nonzero": path.stat().st_size > 0})

required_concepts = {
    "Platonic mesh diagnostics and duality",
    "Interactive Platonic solids and duality",
    "Euler angle-defect budget",
    "Cauchy rigidity sign-change proof state",
    "Semiregular positive-defect vertex configurations",
    "Truncated and semiregular mesh examples",
    "Platonic symmetry group diagnostics",
}
assert required_concepts.issubset({record["concept"] for record in artifact_records})

summary = {
    "unit": UNIT,
    "source_span": "printed pages 435-480; PDF pages 447-492",
    "identity_checks": identity_checks,
    "artifact_count": len(seen_paths),
    "artifacts": artifact_checks,
    "all_paths_book_local": all(record["path"].startswith("artifacts/chapter-08/") for record in artifact_checks),
}
checks_path = save_json(summary, UNIT, "checks", "notebook-sanity.json", root=ARTIFACT_ROOT)
record_artifact(checks_path, "Notebook sanity summary", min_bytes=512)
assert_artifact(checks_path, min_bytes=512)
summary

## Takeaways

- A polyhedron is not just a surface picture. For this chapter it is a mesh
  with incidence, local face data, curvature defect, and symmetry actions.
- The five regular solids are forced by the integer angle-defect budget before
  their coordinates are drawn.
- Euler's formula is the chapter's bookkeeping engine: it certifies meshes,
  powers Descartes' total defect, and appears inside Cauchy's rigidity proof.
- Duality exchanges vertices and faces while preserving edges, which is why
  cube/octahedron and dodecahedron/icosahedron should be studied in pairs.
- Semiregular solids need both local positive defect and metric equality;
  truncation shows why incidence alone is not enough.
- Symmetry groups can be diagnosed by acting on vertices. The coordinate
  counts recover the tetrahedral, octahedral, and icosahedral rotation groups
  used in the source section.